## Re-Ranker: Cross-Encoder on Ambiguous Cases

Questo notebook allena un secondo modello BERT-base che riclassifica
i candidati su cui A5 era incerto (margin basso).

**Pipeline:**
1. A5 produce logits per tutte le coppie
2. I casi con `margin < AMBIGUITY_THRESHOLD` vengono passati al re-ranker
3. Il re-ranker usa **full context** (intero abstract) per decidere

**Vantaggi rispetto ad A6:**
- Training molto più veloce (solo sui casi ambigui)
- Modello base → ~4GB VRAM
- Complementare ad A5/A6: può essere ensemblato

## Configuration

In [1]:
from pathlib import Path

# ── CHANGE THESE ──────────────────────────────────────────────────────────
A5_MODEL_DIR     = Path("models/pubmedbert_large_re_A5_hardneg").resolve()
A5_BASE_MODEL    = "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract"
A5_CACHE_PATH    = Path("models/pubmedbert_large_re_A5_hardneg/dev_logits_cache.pt")

RR_BASE_MODEL    = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
RR_MODEL_DIR     = Path("models/reranker_bert_base_fullctx").resolve()

# Soglia: coppie con margin < threshold vanno al re-ranker
# (margin = p_best - p_no_relation dopo softmax)
AMBIGUITY_THRESHOLD = 0.35
# ──────────────────────────────────────────────────────────────────────────

TRAIN_FILES = [
    "C:/Users/super/Documents/UniPd/ATA/GutBrainIE/data/GutBrainIE_Full_Collection_2026/Annotations/Train/gold_quality/json_format/train_gold.json",
    "C:/Users/super/Documents/UniPd/ATA/GutBrainIE/data/GutBrainIE_Full_Collection_2026/Annotations/Train/silver_quality/json_format/train_silver.json",
    "C:/Users/super/Documents/UniPd/ATA/GutBrainIE/data/GutBrainIE_Full_Collection_2026/Annotations/Train/bronze_quality/json_format/train_bronze.json",
    "C:/Users/super/Documents/UniPd/ATA/GutBrainIE/data/GutBrainIE_Full_Collection_2026/Annotations/Train/silver_quality/json_format/train_silver_2025.json",
]
DEV_PATH = "C:/Users/super/Documents/UniPd/ATA/GutBrainIE/data/GutBrainIE_Full_Collection_2026/Annotations/Dev/json_format/dev.json"
MAX_LENGTH  = 512
BATCH_SIZE  = 4
GRAD_ACCUM  = 8
EPOCHS      = 3
LR          = 2e-5
SEED        = 42
NEG_MULT    = 3   # negativi per positivo nel training del re-ranker


## Imports

In [2]:
import json, os, random, re
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer, AutoModel,
    TrainingArguments, Trainer
)
from dataclasses import dataclass
from transformers import PreTrainedTokenizerBase

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 5070 Laptop GPU


## Labels and Legal Pairs

In [3]:
import re as _re

LEGAL_ENTITY_LABELS = {
    "anatomical location","animal","bacteria","biomedical technique","chemical","DDF",
    "dietary supplement","drug","food","gene","human","microbiome","statistical technique"
}
LEGAL_RELATION_LABELS = {
    "administered","affect","change abundance","change effect","change expression",
    "compared to","impact","influence","interact","is a","is linked to","located in",
    "part of","produced by","strike","target","used by"
}
RELATION_LABELS = [
    "no relation","administered","affect","change abundance","change effect",
    "change expression","compared to","impact","influence","interact","is a",
    "is linked to","located in","part of","produced by","strike","target","used by"
]
label2id = {l: i for i, l in enumerate(RELATION_LABELS)}
id2label = {i: l for i, l in enumerate(RELATION_LABELS)}

LEGAL_RELATIONS = [
    ("DDF","affect","DDF"),("microbiome","is linked to","DDF"),("DDF","target","human"),
    ("drug","change effect","DDF"),("DDF","is a","DDF"),("microbiome","located in","human"),
    ("chemical","influence","DDF"),("dietary supplement","influence","DDF"),("DDF","target","animal"),
    ("chemical","impact","microbiome"),("anatomical location","located in","animal"),
    ("microbiome","located in","animal"),("chemical","located in","anatomical location"),
    ("bacteria","part of","microbiome"),("DDF","strike","anatomical location"),
    ("drug","administered","animal"),("bacteria","influence","DDF"),("drug","impact","microbiome"),
    ("DDF","change abundance","microbiome"),("microbiome","located in","anatomical location"),
    ("microbiome","used by","biomedical technique"),("chemical","produced by","microbiome"),
    ("dietary supplement","impact","microbiome"),("bacteria","located in","animal"),
    ("animal","used by","biomedical technique"),("chemical","impact","bacteria"),
    ("chemical","located in","animal"),("food","impact","bacteria"),
    ("microbiome","compared to","microbiome"),("human","used by","biomedical technique"),
    ("bacteria","change expression","gene"),("chemical","located in","human"),
    ("drug","interact","chemical"),("food","administered","human"),
    ("DDF","change abundance","bacteria"),("chemical","interact","chemical"),
    ("chemical","part of","chemical"),("dietary supplement","impact","bacteria"),
    ("DDF","interact","chemical"),("food","impact","microbiome"),
    ("food","influence","DDF"),("bacteria","located in","human"),
    ("dietary supplement","administered","human"),("bacteria","interact","chemical"),
    ("drug","change expression","gene"),("drug","impact","bacteria"),
    ("drug","administered","human"),("anatomical location","located in","human"),
    ("dietary supplement","change expression","gene"),("chemical","change expression","gene"),
    ("bacteria","interact","bacteria"),("drug","interact","drug"),
    ("microbiome","change expression","gene"),("bacteria","interact","drug"),
    ("food","change expression","gene")
]

def norm_ent(label):
    if label is None: return ""
    lab = str(label).strip()
    return "DDF" if lab.lower() == "ddf" else lab

def norm_span(s):
    return _re.sub(r"\s+", " ", str(s).strip())

legal_pairs = {}
for s, p, o in LEGAL_RELATIONS:
    s = norm_ent(s); o = norm_ent(o)
    legal_pairs.setdefault((s, o), set()).add(p)

print(f"Labels: {len(RELATION_LABELS)}, Legal pairs: {len(legal_pairs)}")


Labels: 18, Legal pairs: 52


## Model Architecture (same as A5)

In [4]:
def entity_average(hidden, mask):
    mask = mask.unsqueeze(-1).float()
    summed = (hidden * mask).sum(dim=1)
    count = mask.sum(dim=1).clamp(min=1e-6)
    return summed / count


class BertForREWithEntityMarkers(nn.Module):
    """
    BERT RE model with mention-mean pooling between markers.
    Identical architecture to A5/A6 — only base model changes.
    """
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size * 2, num_labels)
        self.num_labels = num_labels

    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None):
        self.bert.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs=gradient_checkpointing_kwargs)

    def gradient_checkpointing_disable(self):
        self.bert.gradient_checkpointing_disable()

    def forward(self, input_ids, attention_mask, e1_mask, e2_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq = outputs.last_hidden_state
        e1_h = entity_average(seq, e1_mask)
        e2_h = entity_average(seq, e2_mask)
        concat_h = self.dropout(torch.cat([e1_h, e2_h], dim=-1))
        logits = self.classifier(concat_h)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits.view(-1, self.num_labels), labels.view(-1))
        return {"loss": loss, "logits": logits}


print("Model class defined")


Model class defined


## Load A5 to Identify Ambiguous Cases

Usare i logits cachati di A5 per trovare i training examples su cui A5 era incerto.
Questi sono i casi che il re-ranker deve imparare a gestire.

In [5]:
# Load A5 dev logits cache
print("Loading A5 logits cache...")
dev_cache = torch.load(str(A5_CACHE_PATH), map_location="cpu")
print(f"Documents in cache: {len(dev_cache)}")

total_pairs = sum(len(v) for v in dev_cache.values())
print(f"Total candidate pairs: {total_pairs}")

# Compute margin distribution to calibrate AMBIGUITY_THRESHOLD
def softmax_np(x):
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / ex.sum()

margins = []
for pmid, rows in dev_cache.items():
    for row in rows:
        logits = row["logits"].numpy()
        probs  = softmax_np(logits)
        p_no   = float(probs[label2id["no relation"]])
        s_lab, o_lab = row["subject_label"], row["object_label"]
        allowed = legal_pairs.get((s_lab, o_lab), [])
        if not allowed:
            continue
        allowed_ids = [label2id[p] for p in allowed]
        p_best = float(max(probs[i] for i in allowed_ids))
        margins.append(p_best - p_no)

margins = np.array(margins)
print(f"\nMargin stats:")
print(f"  mean={margins.mean():.3f}  median={np.median(margins):.3f}")
print(f"  < 0.10: {(margins < 0.10).mean()*100:.1f}%")
print(f"  < 0.20: {(margins < 0.20).mean()*100:.1f}%")
print(f"  < 0.35: {(margins < 0.35).mean()*100:.1f}%")
print(f"  < 0.50: {(margins < 0.50).mean()*100:.1f}%")
print(f"\nWith AMBIGUITY_THRESHOLD={AMBIGUITY_THRESHOLD}: "
      f"{(margins < AMBIGUITY_THRESHOLD).mean()*100:.1f}% of pairs go to re-ranker")


Loading A5 logits cache...
Documents in cache: 80
Total candidate pairs: 50202

Margin stats:
  mean=-0.030  median=0.067
  < 0.10: 50.5%
  < 0.20: 51.9%
  < 0.35: 54.2%
  < 0.50: 57.0%

With AMBIGUITY_THRESHOLD=0.35: 54.2% of pairs go to re-ranker


## Load Training Data

In [6]:
def load_re_data(file_paths):
    all_data = {}
    for p in file_paths:
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                data = json.load(f)
            all_data.update(data)
            print(f"  Loaded {len(data)} docs from {os.path.basename(p)}")
        else:
            print(f"  WARNING: not found: {p}")
    return all_data

print("Loading training data...")
train_data = load_re_data(TRAIN_FILES)
print(f"Total train docs: {len(train_data)}")

print("\nLoading dev data...")
dev_data = load_re_data([DEV_PATH])
print(f"Total dev docs: {len(dev_data)}")


Loading training data...
Total train docs: 0

Loading dev data...
Total dev docs: 0


## Build Re-Ranker Training Examples

Strategia:
- **Positivi**: tutti gli esempi positivi del training (il re-ranker deve sapere riconoscere le relazioni)
- **Negativi hard**: coppie che A5 confonde — con tipi di entità compatibili ma senza relazione
- Focus su predicati rari (`strike`, `used by`, `part of`, `change expression`) che A5 sbaglia di più

In [7]:
def create_full_text_with_offsets(title, abstract):
    full_text = f"{title} {abstract}"
    return full_text, len(title) + 1

def adjust_entity_positions(entity, abstract_offset):
    if entity["location"] == "abstract":
        return {**entity,
                "start_idx": entity["start_idx"] + abstract_offset,
                "end_idx":   entity["end_idx"]   + abstract_offset}
    return dict(entity)

# Predicati rari: peso doppio nel campionamento
RARE_PREDICATES = {"strike", "used by", "part of", "change expression", "change abundance"}

def prepare_reranker_examples(data, neg_multiplier=3, legal_pairs=None):
    examples = []

    for pmid, article in tqdm(data.items(), desc="Building re-ranker examples"):
        title    = article["metadata"]["title"]
        abstract = article["metadata"]["abstract"]
        full_text, abs_offset = create_full_text_with_offsets(title, abstract)

        entities = [
            {
                **adjust_entity_positions(e, abs_offset),
                "label":     norm_ent(e["label"]),
                "text_span": norm_span(e["text_span"]),
            }
            for e in article["entities"]
        ]

        relations = article.get("mention_level_relations", [])

        ent_index = defaultdict(list)
        for e in entities:
            ent_index[(e["text_span"], e["label"])].append(e)

        positive_keys = set()

        # ── Positives ──────────────────────────────────────────────────────
        for rel in relations:
            pred     = rel["predicate"].strip()
            s_text   = norm_span(rel["subject_text_span"])
            o_text   = norm_span(rel["object_text_span"])
            s_lab    = norm_ent(rel["subject_label"])
            o_lab    = norm_ent(rel["object_label"])

            if pred not in LEGAL_RELATION_LABELS: continue
            if s_lab not in LEGAL_ENTITY_LABELS or o_lab not in LEGAL_ENTITY_LABELS: continue
            if legal_pairs and (s_lab, o_lab) not in legal_pairs: continue

            s_cands = ent_index.get((s_text, s_lab), [])
            o_cands = ent_index.get((o_text, o_lab), [])
            if not s_cands or not o_cands: continue

            # pick closest pair
            best_s, best_o, best_dist = None, None, float("inf")
            for s in s_cands:
                for o in o_cands:
                    if s["start_idx"] == o["start_idx"]: continue
                    d = abs(s["start_idx"] - o["start_idx"])
                    if d < best_dist:
                        best_dist = d; best_s = s; best_o = o
            if best_s is None: continue

            n_copies = 2 if pred in RARE_PREDICATES else 1
            for _ in range(n_copies):
                examples.append({
                    "text": full_text, "subject": best_s, "object": best_o,
                    "predicate": pred, "pmid": pmid
                })
            positive_keys.add((best_s["start_idx"], best_s["end_idx"],
                                best_o["start_idx"], best_o["end_idx"]))

        # ── Hard Negatives ─────────────────────────────────────────────────
        n_neg = len(positive_keys) * neg_multiplier
        neg_cands = []
        for i, s in enumerate(entities):
            for j, o in enumerate(entities):
                if i == j: continue
                if legal_pairs and (s["label"], o["label"]) not in legal_pairs: continue
                key = (s["start_idx"], s["end_idx"], o["start_idx"], o["end_idx"])
                if key in positive_keys: continue
                neg_cands.append({
                    "text": full_text, "subject": s, "object": o,
                    "predicate": "no relation", "pmid": pmid
                })

        if neg_cands:
            sampled = random.sample(neg_cands, min(n_neg, len(neg_cands)))
            examples.extend(sampled)

    return examples


print("Preparing re-ranker training examples...")
train_examples = prepare_reranker_examples(
    train_data, neg_multiplier=NEG_MULT, legal_pairs=legal_pairs
)
print("Preparing re-ranker dev examples...")
dev_examples = prepare_reranker_examples(
    dev_data, neg_multiplier=NEG_MULT, legal_pairs=legal_pairs
)

pos_tr = sum(1 for e in train_examples if e["predicate"] != "no relation")
neg_tr = len(train_examples) - pos_tr
print(f"\nTrain: {len(train_examples)} examples  (pos={pos_tr}, neg={neg_tr}, ratio={neg_tr/max(pos_tr,1):.1f})")
print(f"Dev:   {len(dev_examples)} examples")


Preparing re-ranker training examples...


Building re-ranker examples: 0it [00:00, ?it/s]


Preparing re-ranker dev examples...


Building re-ranker examples: 0it [00:00, ?it/s]


Train: 0 examples  (pos=0, neg=0, ratio=0.0)
Dev:   0 examples


## Initialize Tokenizer

In [8]:
print("Loading tokenizer:", RR_BASE_MODEL)
tokenizer = AutoTokenizer.from_pretrained(RR_BASE_MODEL, use_fast=True)

special_tokens = {"additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]}
tokenizer.add_special_tokens(special_tokens)

e1_token_id  = tokenizer.convert_tokens_to_ids("[E1]")
e1e_token_id = tokenizer.convert_tokens_to_ids("[/E1]")
e2_token_id  = tokenizer.convert_tokens_to_ids("[E2]")
e2e_token_id = tokenizer.convert_tokens_to_ids("[/E2]")

print(f"Vocab size: {len(tokenizer)}")
print(f"[E1]={e1_token_id}  [/E1]={e1e_token_id}  [E2]={e2_token_id}  [/E2]={e2e_token_id}")


Loading tokenizer: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext


C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\super\.cache\huggingface\hub\models--microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Vocab size: 30526
[E1]=30522  [/E1]=30523  [E2]=30524  [/E2]=30525


## Tokenization — Full Context + Mention-Mean

In [9]:
def insert_entity_markers(text, subject, obj):
    entities = sorted([
        (subject["start_idx"], subject["end_idx"], "[E1]", "[/E1]"),
        (obj["start_idx"],     obj["end_idx"],     "[E2]", "[/E2]"),
    ], key=lambda x: x[0])
    marked, offset = text, 0
    for start, end, sm, em in entities:
        a, b = start + offset, end + offset + 1
        marked = marked[:a] + sm + marked[a:b] + em + marked[b:]
        offset += len(sm) + len(em)
    return marked


def span_mask_between_markers(input_ids, start_tok_id, end_tok_id):
    """Mask = 1 for tokens BETWEEN the marker pair (mention-mean pooling)."""
    mask = torch.zeros_like(input_ids)
    in_span = False
    for i, tok in enumerate(input_ids.tolist()):
        if tok == start_tok_id:  in_span = True;  continue
        if tok == end_tok_id:    in_span = False;  continue
        if in_span:              mask[i] = 1
    return mask


def tokenize_example(example, tokenizer, e1_start, e1_end, e2_start, e2_end,
                     max_length=512):
    """Full-context tokenization with mention-mean masks."""
    marked = insert_entity_markers(
        example["text"], example["subject"], example["object"]
    )
    enc = tokenizer(
        marked, truncation=True, max_length=max_length,
        padding=False, return_tensors="pt"
    )
    input_ids      = enc["input_ids"].squeeze(0)
    attention_mask = enc["attention_mask"].squeeze(0)

    e1_mask = span_mask_between_markers(input_ids, e1_start, e1_end)
    e2_mask = span_mask_between_markers(input_ids, e2_start, e2_end)

    if e1_mask.sum() == 0 or e2_mask.sum() == 0:
        return None

    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "e1_mask":        e1_mask,
        "e2_mask":        e2_mask,
        "labels":         torch.tensor(label2id[example["predicate"]], dtype=torch.long),
    }


print("Tokenization functions defined")


Tokenization functions defined


## Pre-tokenize and Cache

In [10]:
CACHE_DIR = RR_MODEL_DIR / "cache_tok"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_CACHE = str(CACHE_DIR / f"rr_train_maxlen{MAX_LENGTH}.pt")
DEV_CACHE   = str(CACHE_DIR / f"rr_dev_maxlen{MAX_LENGTH}.pt")


class ListREDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]


def build_dataset(examples, cache_path):
    if os.path.exists(cache_path):
        print(f"[cache] Loading: {cache_path}")
        payload = torch.load(cache_path, map_location="cpu")
        return ListREDataset(payload["items"]), payload.get("skipped", 0)

    items, skipped = [], 0
    for ex in tqdm(examples, desc="Tokenizing"):
        out = tokenize_example(
            ex, tokenizer,
            e1_token_id, e1e_token_id,
            e2_token_id, e2e_token_id,
            max_length=MAX_LENGTH
        )
        if out is None: skipped += 1; continue
        items.append(out)

    torch.save({"items": items, "skipped": skipped}, cache_path)
    print(f"[cache] Saved {len(items)} items ({skipped} skipped) → {cache_path}")
    return ListREDataset(items), skipped


print("Building train dataset...")
train_dataset, tr_skip = build_dataset(train_examples, TRAIN_CACHE)
print("Building dev dataset...")
dev_dataset,   dv_skip = build_dataset(dev_examples,   DEV_CACHE)

print(f"\nTrain: {len(train_dataset)} items ({tr_skip} skipped)")
print(f"Dev:   {len(dev_dataset)} items ({dv_skip} skipped)")


Building train dataset...


Tokenizing: 0it [00:00, ?it/s]


[cache] Saved 0 items (0 skipped) → C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\re\models\reranker_bert_base_fullctx\cache_tok\rr_train_maxlen512.pt
Building dev dataset...


Tokenizing: 0it [00:00, ?it/s]

[cache] Saved 0 items (0 skipped) → C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\re\models\reranker_bert_base_fullctx\cache_tok\rr_dev_maxlen512.pt

Train: 0 items (0 skipped)
Dev:   0 items (0 skipped)


## Data Collator

In [11]:
@dataclass
class REDataCollatorWithPadding:
    tokenizer: PreTrainedTokenizerBase
    pad_to_multiple_of: int = 8

    def __call__(self, features):
        labels = torch.stack([f["labels"] for f in features])
        batch  = self.tokenizer.pad(
            [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]}
             for f in features],
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        max_len = batch["input_ids"].shape[1]

        def pad_1d(x):
            if x.shape[0] == max_len: return x
            out = torch.zeros(max_len, dtype=x.dtype)
            out[:x.shape[0]] = x
            return out

        batch["e1_mask"] = torch.stack([pad_1d(f["e1_mask"]) for f in features])
        batch["e2_mask"] = torch.stack([pad_1d(f["e2_mask"]) for f in features])
        batch["labels"]  = labels
        return batch


collator = REDataCollatorWithPadding(tokenizer=tokenizer)
print("Collator ready")


Collator ready


## Initialize Re-Ranker Model

In [12]:
print("Initializing re-ranker model:", RR_BASE_MODEL)
model = BertForREWithEntityMarkers(RR_BASE_MODEL, num_labels=len(RELATION_LABELS))
model.bert.resize_token_embeddings(len(tokenizer))

print(f"  Hidden size: {model.bert.config.hidden_size}")
print(f"  Num labels:  {model.num_labels}")
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable params: {n_params/1e6:.1f}M")


Initializing re-ranker model: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33763.46it/s]
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialize

  Hidden size: 768
  Num labels:  18
  Trainable params: 109.5M


## Metrics

In [13]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    pos_ids = [i for l, i in label2id.items() if l != "no relation"]
    macro_f1 = f1_score(labels, preds, labels=pos_ids, average="macro",  zero_division=0)
    micro_f1 = f1_score(labels, preds, labels=pos_ids, average="micro",  zero_division=0)
    return {"macro_f1_pos": macro_f1, "micro_f1_pos": micro_f1}

print("compute_metrics defined")


compute_metrics defined


## Custom Trainer

In [14]:
class RETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs, labels=labels)
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

    def _save(self, output_dir, state_dict=None):
        os.makedirs(output_dir, exist_ok=True)
        if state_dict is None:
            state_dict = self.model.state_dict()
        for k, v in state_dict.items():
            if isinstance(v, torch.Tensor) and not v.is_contiguous():
                state_dict[k] = v.contiguous()
        torch.save(state_dict, os.path.join(output_dir, "pytorch_model.bin"))
        torch.save(self.args,  os.path.join(output_dir, "training_args.bin"))

print("RETrainer defined")


RETrainer defined


## Training Arguments

In [15]:
training_args = TrainingArguments(
    output_dir=str(RR_MODEL_DIR),

    learning_rate=LR,
    warmup_ratio=0.06,
    lr_scheduler_type="linear",

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    max_grad_norm=1.0,

    num_train_epochs=EPOCHS,
    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_micro_f1_pos",
    greater_is_better=True,

    fp16=False,
    bf16=torch.cuda.is_available(),
    tf32=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    logging_steps=50,
    disable_tqdm=False,

    seed=SEED,
    report_to="none"
)

print("Training args ready")
print(f"  Batch size:  {BATCH_SIZE} x grad_accum {GRAD_ACCUM} = effective {BATCH_SIZE*GRAD_ACCUM}")
print(f"  Epochs:      {EPOCHS}")
print(f"  LR:          {LR}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training args ready
  Batch size:  4 x grad_accum 8 = effective 32
  Epochs:      3
  LR:          2e-05


## Train

In [16]:
trainer = RETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Dev samples:   {len(dev_dataset)}")

import time
t0 = time.time()
print("=" * 60)
print("Starting re-ranker training...")
print("=" * 60)
trainer.train()
print(f"Training time: {(time.time()-t0)/60:.1f} minutes")


Train samples: 0
Dev samples:   0
Starting re-ranker training...


ValueError: num_samples should be a positive integer value, but got num_samples=0

## Save Re-Ranker

In [ ]:
RR_MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(RR_MODEL_DIR))
tokenizer.save_pretrained(str(RR_MODEL_DIR))

with open(RR_MODEL_DIR / "label_mappings.json", "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=2)

# Save config for inference
rr_config = {
    "base_model":           RR_BASE_MODEL,
    "ambiguity_threshold":  AMBIGUITY_THRESHOLD,
    "pooling_mode":         "mention-mean",
    "a5_model_dir":         str(A5_MODEL_DIR),
}
with open(RR_MODEL_DIR / "reranker_config.json", "w") as f:
    json.dump(rr_config, f, indent=2)

print(f"Re-ranker saved to: {RR_MODEL_DIR}")
print(f"Config saved: {RR_MODEL_DIR / 'reranker_config.json'}")


## Inference: A5 + Re-Ranker Pipeline

Logica:
- A5 processa tutte le coppie e produce logits
- Se `margin(A5) >= AMBIGUITY_THRESHOLD` → usa la predizione di A5
- Se `margin(A5) < AMBIGUITY_THRESHOLD` → ri-codifica con full context e usa il re-ranker

In [ ]:
# ── Load A5 model ──────────────────────────────────────────────────────────
def find_last_checkpoint(root_dir):
    root_dir = Path(root_dir)
    ckpts = [d for d in root_dir.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")]
    return sorted(ckpts, key=lambda x: int(x.name.split("-")[1]))[-1] if ckpts else root_dir

print("Loading A5 model...")
a5_tok = AutoTokenizer.from_pretrained(str(A5_MODEL_DIR), use_fast=True, local_files_only=True)
a5_model = BertForREWithEntityMarkers(A5_BASE_MODEL, num_labels=len(RELATION_LABELS))
a5_model.bert.resize_token_embeddings(len(a5_tok))
a5_state = torch.load(
    find_last_checkpoint(A5_MODEL_DIR) / "pytorch_model.bin", map_location="cpu"
)
a5_model.load_state_dict(a5_state)
a5_model.eval().to(DEVICE)

# ── Load Re-Ranker ─────────────────────────────────────────────────────────
print("Loading re-ranker model...")
rr_tok = AutoTokenizer.from_pretrained(str(RR_MODEL_DIR), use_fast=True, local_files_only=True)
rr_e1  = rr_tok.convert_tokens_to_ids("[E1]")
rr_e1e = rr_tok.convert_tokens_to_ids("[/E1]")
rr_e2  = rr_tok.convert_tokens_to_ids("[E2]")
rr_e2e = rr_tok.convert_tokens_to_ids("[/E2]")
rr_model = BertForREWithEntityMarkers(RR_BASE_MODEL, num_labels=len(RELATION_LABELS))
rr_model.bert.resize_token_embeddings(len(rr_tok))
rr_state = torch.load(
    find_last_checkpoint(RR_MODEL_DIR) / "pytorch_model.bin", map_location="cpu"
)
rr_model.load_state_dict(rr_state)
rr_model.eval().to(DEVICE)

print("Both models loaded")


In [ ]:
def softmax_t(logits, temperature=1.0):
    x = logits.float() / temperature
    x = x - x.max()
    ex = torch.exp(x)
    return ex / ex.sum()


@torch.no_grad()
def predict_combined(
    dev_data, a5_model, a5_tok, rr_model, rr_tok,
    legal_pairs, label2id, id2label,
    ambiguity_threshold=AMBIGUITY_THRESHOLD,
    a5_temperature=1.25, rr_temperature=1.0,
    a5_min_prob=0.10, a5_margin=0.25,
    batch_size=16, max_length=512,
):
    a5_e1  = a5_tok.convert_tokens_to_ids("[E1]")
    a5_e1e = a5_tok.convert_tokens_to_ids("[/E1]")
    a5_e2  = a5_tok.convert_tokens_to_ids("[E2]")
    a5_e2e = a5_tok.convert_tokens_to_ids("[/E2]")

    pred_by_doc = {}
    stats = {"a5_decided": 0, "rr_decided": 0, "total": 0}

    for pmid, article in tqdm(dev_data.items(), desc="Combined inference"):
        title    = article["metadata"]["title"]
        abstract = article["metadata"]["abstract"]
        full_text, abs_off = create_full_text_with_offsets(title, abstract)

        adjusted = [
            {**adjust_entity_positions(e, abs_off),
             "label": norm_ent(e["label"]), "text_span": norm_span(e["text_span"])}
            for e in article["entities"]
        ]

        doc_preds = set()

        pairs = [
            (s, o)
            for i, s in enumerate(adjusted)
            for j, o in enumerate(adjusted)
            if i != j and (s["label"], o["label"]) in legal_pairs
        ]

        if not pairs:
            pred_by_doc[str(pmid)] = doc_preds
            continue

        # ── A5 pass ───────────────────────────────────────────────────────
        ambiguous = []
        for s, o in pairs:
            stats["total"] += 1
            marked = insert_entity_markers(full_text, s, o)
            enc = a5_tok(marked, truncation=True, max_length=max_length,
                         padding=False, return_tensors="pt")
            input_ids = enc["input_ids"].squeeze(0)
            attn      = enc["attention_mask"].squeeze(0)
            e1m = span_mask_between_markers(input_ids, a5_e1, a5_e1e)
            e2m = span_mask_between_markers(input_ids, a5_e2, a5_e2e)
            if e1m.sum() == 0 or e2m.sum() == 0: continue

            out = a5_model(
                input_ids=input_ids.unsqueeze(0).to(DEVICE),
                attention_mask=attn.unsqueeze(0).to(DEVICE),
                e1_mask=e1m.unsqueeze(0).to(DEVICE),
                e2_mask=e2m.unsqueeze(0).to(DEVICE),
            )
            logits = out["logits"].squeeze(0).cpu()
            probs  = softmax_t(logits, a5_temperature)

            allowed = sorted(legal_pairs.get((s["label"], o["label"]), []))
            allowed_ids = [label2id["no relation"]] + [label2id[p] for p in allowed]
            sub_probs   = softmax_t(logits[allowed_ids])
            p_no        = float(sub_probs[0])
            rel_probs   = sub_probs[1:]
            best_idx    = int(rel_probs.argmax())
            pred_label  = allowed[best_idx]
            p_best      = float(rel_probs[best_idx])
            margin      = p_best - p_no

            if margin >= ambiguity_threshold and p_best >= a5_min_prob and margin >= a5_margin:
                # A5 is confident → use its prediction
                doc_preds.add((norm_span(s["text_span"]), s["label"],
                               pred_label,
                               norm_span(o["text_span"]), o["label"]))
                stats["a5_decided"] += 1
            else:
                # Ambiguous → queue for re-ranker
                ambiguous.append((s, o, pred_label, p_best, margin))

        # ── Re-ranker pass ────────────────────────────────────────────────
        for s, o, a5_pred, a5_p, a5_m in ambiguous:
            marked = insert_entity_markers(full_text, s, o)
            enc = rr_tok(marked, truncation=True, max_length=max_length,
                         padding=False, return_tensors="pt")
            input_ids = enc["input_ids"].squeeze(0)
            attn      = enc["attention_mask"].squeeze(0)
            e1m = span_mask_between_markers(input_ids, rr_e1, rr_e1e)
            e2m = span_mask_between_markers(input_ids, rr_e2, rr_e2e)
            if e1m.sum() == 0 or e2m.sum() == 0: continue

            out = rr_model(
                input_ids=input_ids.unsqueeze(0).to(DEVICE),
                attention_mask=attn.unsqueeze(0).to(DEVICE),
                e1_mask=e1m.unsqueeze(0).to(DEVICE),
                e2_mask=e2m.unsqueeze(0).to(DEVICE),
            )
            logits = out["logits"].squeeze(0).cpu()

            allowed = sorted(legal_pairs.get((s["label"], o["label"]), []))
            allowed_ids = [label2id["no relation"]] + [label2id[p] for p in allowed]
            sub_probs   = softmax_t(logits[allowed_ids], rr_temperature)
            p_no        = float(sub_probs[0])
            rel_probs   = sub_probs[1:]
            best_idx    = int(rel_probs.argmax())
            pred_label  = allowed[best_idx]
            p_best      = float(rel_probs[best_idx])
            rr_margin   = p_best - p_no

            if rr_margin > 0.05:  # re-ranker mildly confident
                doc_preds.add((norm_span(s["text_span"]), s["label"],
                               pred_label,
                               norm_span(o["text_span"]), o["label"]))
                stats["rr_decided"] += 1

        pred_by_doc[str(pmid)] = doc_preds

    print(f"\nStats: total={stats['total']}  "
          f"a5_decided={stats['a5_decided']}  "
          f"rr_decided={stats['rr_decided']}")
    return pred_by_doc


print("Combined inference function defined")


## Evaluate Combined Pipeline

In [ ]:
# Build gold labels for dev
gold_by_doc = {}
for pmid, article in dev_data.items():
    gold_set = set()
    for rel in article.get("mention_level_relations", []):
        pred = rel["predicate"].strip()
        if pred not in LEGAL_RELATION_LABELS: continue
        gold_set.add((
            norm_span(rel["subject_text_span"]), norm_ent(rel["subject_label"]),
            pred,
            norm_span(rel["object_text_span"]),  norm_ent(rel["object_label"])
        ))
    gold_by_doc[str(pmid)] = gold_set

def micro_scores(gold, pred):
    tp = fp = fn = 0
    for pmid in gold:
        g = gold.get(pmid, set())
        p = pred.get(pmid, set())
        tp += len(g & p); fp += len(p - g); fn += len(g - p)
    P = tp / (tp + fp) if tp + fp > 0 else 0
    R = tp / (tp + fn) if tp + fn > 0 else 0
    F = 2*P*R / (P+R) if P+R > 0 else 0
    return {"P": P, "R": R, "F1": F}

print("Running combined A5 + Re-Ranker inference on dev...")
pred_by_doc = predict_combined(
    dev_data, a5_model, a5_tok, rr_model, rr_tok,
    legal_pairs, label2id, id2label,
)

sc = micro_scores(gold_by_doc, pred_by_doc)
print(f"\n*** COMBINED PIPELINE — Dev Micro F1: {sc['F1']:.4f} ***")
print(f"    P={sc['P']:.4f}  R={sc['R']:.4f}")
